一个tokenizer网址：https://tiktokenizer.vercel.app/
从视频中可以看到，以GPT2为例，数学加法中的数字可能会被拆分；同一单词可能因为大小写不同而被以不一样的方式拆分；非英文文本token数被拉长了，有许多小的边界，是由于训练数据中英文文本占多数，与用于tokenizer和分词本身的训练有关
而对于GPT4的tokenizer，其token数是GPT2的两倍（embedding table也会变大，后续的softmax会增加计算量），那么同样的文本其token表示可以被压缩到一半

ord()只接受单个字符，不能直接就用这个来进行tokenize，原因：

1、过长

2、Unicode一直变化，不是稳定表示

In [9]:
[ord(x) for x in "안녕하세요 👋 (hello in Korean!)"]

[50504,
 45397,
 54616,
 49464,
 50836,
 32,
 128075,
 32,
 40,
 104,
 101,
 108,
 108,
 111,
 32,
 105,
 110,
 32,
 75,
 111,
 114,
 101,
 97,
 110,
 33,
 41]

直接使用UTF-8也不好，因为这会将输入文本变为很长的字节序列，但transformer的上下文长度有限，不利于关注本身的输入文本（都集中在了字节序列上）。

所希望的是能够支持更大的词汇量，并将其作为超参进行调整

In [10]:
list("안녕하세요 👋 (hello in Korean!)".encode("utf-8"))

[236,
 149,
 136,
 235,
 133,
 149,
 237,
 149,
 152,
 236,
 132,
 184,
 236,
 154,
 148,
 32,
 240,
 159,
 145,
 139,
 32,
 40,
 104,
 101,
 108,
 108,
 111,
 32,
 105,
 110,
 32,
 75,
 111,
 114,
 101,
 97,
 110,
 33,
 41]

### Byte pair encoding

找到出现频率最高的两个token对，一旦确定了这个对，就用其替代所有文本中的这个对，重复这个过程直到达到预定的词汇量

https://en.wikipedia.org/wiki/Byte-pair_encoding

In [11]:
# text from https://www.reedbeta.com/blog/programmers-intro-to-unicode/
text = "Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception."
tokens = text.encode("utf-8") # raw bytes
tokens = list(map(int, tokens)) # convert to a list of integers in range 0..255 for convenience
print('---')
print(text)
print("length:", len(text))
print('---')
print(tokens)
print("length:", len(tokens))

---
Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception.
length: 533
---
[239, 188, 181, 239, 189, 142, 239, 189, 137, 239, 189, 131, 239, 189, 143, 239, 189, 132, 239, 189, 133, 33, 32, 240, 159, 133, 164, 240, 159, 133, 157, 240, 159, 133, 152, 240, 159, 133, 146, 240, 159, 133, 158, 240, 159, 133, 147, 240, 159, 133, 148, 226, 128, 189, 32, 240, 159, 135, 186, 226, 128, 140, 240, 159, 135, 179, 226, 128, 140, 240, 159, 135, 174, 226, 128, 140, 240, 159, 135, 168, 226, 128, 140, 240, 159, 135, 180, 226, 128, 140

操作：找最频繁的字节对，合并

zip函数作用：将多个可迭代对象按位置配对打包，返回迭代器。例如;

'''python
a = [1, 2, 3]
b = ['a', 'b', 'c']

list(zip(a, b))

结果: [(1, 'a'), (2, 'b'), (3, 'c')]
'''

In [12]:
def get_stats(ids):
    count = {}
    for pair in zip(ids, ids[1:]):  # 通过切片错位，实现相邻元素配对
        count[pair] = count.get(pair, 0) + 1
    return count

stats = get_stats(tokens)

In [13]:
top_pair = max(stats, key=stats.get)  # 字典的 .get() 方法，用于获取键对应的值
top_pair

(101, 32)

In [14]:
def merge(ids, pair, idx):  # 找到ids中所有连续出现的pair，并用idx替换它们
  # in the list of ints (ids), replace all consecutive occurences of pair with the new token idx
  newids = []
  i = 0
  while i < len(ids):
    # if we are not at the very last position AND the pair matches, replace it
    if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:  # 找到连续出现的pair
      newids.append(idx)
      i += 2
    else:
      newids.append(ids[i])  # 否则，直接添加原来的token
      i += 1
  return newids

print(merge([5, 6, 6, 7, 9, 1], (6, 7), 99))

tokens2 = merge(tokens, top_pair, 256)
print(tokens2)
print("length:", len(tokens2))

[5, 6, 99, 9, 1]
[239, 188, 181, 239, 189, 142, 239, 189, 137, 239, 189, 131, 239, 189, 143, 239, 189, 132, 239, 189, 133, 33, 32, 240, 159, 133, 164, 240, 159, 133, 157, 240, 159, 133, 152, 240, 159, 133, 146, 240, 159, 133, 158, 240, 159, 133, 147, 240, 159, 133, 148, 226, 128, 189, 32, 240, 159, 135, 186, 226, 128, 140, 240, 159, 135, 179, 226, 128, 140, 240, 159, 135, 174, 226, 128, 140, 240, 159, 135, 168, 226, 128, 140, 240, 159, 135, 180, 226, 128, 140, 240, 159, 135, 169, 226, 128, 140, 240, 159, 135, 170, 33, 32, 240, 159, 152, 132, 32, 84, 104, 256, 118, 101, 114, 121, 32, 110, 97, 109, 256, 115, 116, 114, 105, 107, 101, 115, 32, 102, 101, 97, 114, 32, 97, 110, 100, 32, 97, 119, 256, 105, 110, 116, 111, 32, 116, 104, 256, 104, 101, 97, 114, 116, 115, 32, 111, 102, 32, 112, 114, 111, 103, 114, 97, 109, 109, 101, 114, 115, 32, 119, 111, 114, 108, 100, 119, 105, 100, 101, 46, 32, 87, 256, 97, 108, 108, 32, 107, 110, 111, 119, 32, 119, 256, 111, 117, 103, 104, 116, 32, 116, 111, 

至于词汇表大小，是一个超参数，由自己进行设置和平衡

In [15]:
# ---
vocab_size = 276 # 刚好进行20次merge，256 + 20 = 276
num_merges = vocab_size - 256
ids = list(tokens) # copy so we don't destroy the original list

merges = {} # (int, int) -> int，维护这个合并关系
for i in range(num_merges):
  stats = get_stats(ids)
  pair = max(stats, key=stats.get)  # 选择出现次数最多的pair进行合并
  idx = 256 + i
  print(f"merging {pair} into a new token {idx}")
  ids = merge(ids, pair, idx)
  merges[pair] = idx

merging (101, 32) into a new token 256
merging (240, 159) into a new token 257
merging (226, 128) into a new token 258
merging (105, 110) into a new token 259
merging (115, 32) into a new token 260
merging (97, 110) into a new token 261
merging (116, 104) into a new token 262
merging (257, 133) into a new token 263
merging (257, 135) into a new token 264
merging (97, 114) into a new token 265
merging (239, 189) into a new token 266
merging (258, 140) into a new token 267
merging (267, 264) into a new token 268
merging (101, 114) into a new token 269
merging (111, 114) into a new token 270
merging (116, 32) into a new token 271
merging (259, 103) into a new token 272
merging (115, 116) into a new token 273
merging (261, 100) into a new token 274
merging (32, 262) into a new token 275


In [16]:
print("tokens length:", len(tokens))
print("ids length:", len(ids))
print(f"compression ratio: {len(tokens) / len(ids):.2f}X")

tokens length: 616
ids length: 451
compression ratio: 1.37X


tokenize和LLM是两个完全独立的对象，需要独立分开训练。tokenizer训练集中不同语言的数量将决定其合并的数量，从而决定这种数据在token空间中的密度

1、第一步，tokenize预处理

2、第二步，获取到转换后的token sequence，LLM进行读取，而不是原始序列

## decoding

In [18]:
vocab = {idx:bytes([idx]) for idx in range(256)} # 0..255 are bytes
for (p0, p1), idx in merges.items():  # 按插入顺序进行合并
  vocab[idx] = vocab[p0] + vocab[p1]  # 两个字节对象的拼接


def decode(ids):
  tokens = b"".join(vocab[idx] for idx in ids)  # 将ids中的每个token替换为对应的字节对象，并拼接成一个大的字节对象
  text = tokens.decode("utf-8", errors="replace")  # 将字节对象解码为字符串，并替换非法字符，否则默认的errors设置下无法正常解码
  return text



## encoding
获得字符串，编码为tokens

In [19]:
def encode(text):
  tokens = list(text.encode("utf-8"))  # 将字符串编码为字节对象
  while len(tokens) > 1:  # 若只有1个字符或空字符，就没必要合并
    stats = get_stats(tokens)  # 获取当前 token 序列中所有相邻字符对的出现频率统计
    pair = min(stats, key=lambda p: merges.get(p, float("inf")))  # 选择最早被合并的pair进行替换
    if pair not in merges:  # 如果没有更多的pair可以合并了，退出循环
      break
    idx = merges[pair]  # 获取合并后的新token的id
    tokens = merge(tokens, pair, idx)  # 将所有连续出现的pair替
  return tokens

print(encode(" hello world!"))

[32, 104, 101, 108, 108, 111, 32, 119, 270, 108, 100, 33]


In [20]:
print(decode(encode(" hello world!")))

 hello world!


### 使用regex进行tokenize



In [21]:
import regex as re
gpt2pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d|?\p{L}+|?\p{N}+|?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

print(re.findall(gpt2pat, "Hello World")) # 将文本分割成token列表，符合GPT-2的tokenization规则

error: nothing to repeat at position 24

In [22]:
import tiktoken

# GPT2
enc = tiktoken.get_encoding("gpt2")
print(enc.encode("    hello world!!!"))

# GPT4
enc = tiktoken.get_encoding("cl100k_base")
print(enc.encode("    hello world!!!"))

[220, 220, 220, 23748, 995, 10185]
[262, 24748, 1917, 12340]


In [14]:
!wget https://openaipublic.blob.core.windows.net/gpt-2/models/1558M/vocab.bpe
!wget https://openaipublic.blob.core.windows.net/gpt-2/models/1558M/encoder.json

--2026-03-11 15:51:27--  https://openaipublic.blob.core.windows.net/gpt-2/models/1558M/vocab.bpe
Resolving openaipublic.blob.core.windows.net (openaipublic.blob.core.windows.net)... 20.60.244.1
Connecting to openaipublic.blob.core.windows.net (openaipublic.blob.core.windows.net)|20.60.244.1|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 456318 (446K) [application/octet-stream]
Saving to: ‘vocab.bpe’

vocab.bpe           100%[===================>] 445.62K   331KB/s    in 1.3s    

2026-03-11 15:51:30 (331 KB/s) - ‘vocab.bpe’ saved [456318/456318]

--2026-03-11 15:51:31--  https://openaipublic.blob.core.windows.net/gpt-2/models/1558M/encoder.json
Resolving openaipublic.blob.core.windows.net (openaipublic.blob.core.windows.net)... 20.60.244.1
Connecting to openaipublic.blob.core.windows.net (openaipublic.blob.core.windows.net)|20.60.244.1|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1042301 (1018K) [application/json]
Saving to: ‘enco

In [23]:
import os, json

with open('encoder.json', 'r') as f:
    encoder = json.load(f) # <--- ~equivalent to our "vocab"

with open('vocab.bpe', 'r', encoding="utf-8") as f:
    bpe_data = f.read()
bpe_merges = [tuple(merge_str.split()) for merge_str in bpe_data.split('\n')[1:-1]]
# ^---- ~equivalent to our "merges"

### special tokens
这个标记意味着应当清除之前的内容记忆，因为与接下来的内容不相关，比如说用于分隔模型与用户之间的对话

**添加特殊tokens**

1、修改Transformers的相关参数：添加一个整数，确保我的tokens的嵌入矩阵能够添加一行进行扩展，来代表该token的向量

2、进入Transformer的最后一层，在分类器的投影头里也增加一个

In [24]:
len(encoder)  # 256个原始tokens，50000个合并tokens，1个special token

50257

In [25]:
encoder['<|endoftext|>']  # special token，表示文本结束，用于在训练集中分割文档

50256

### SentencePiece

In [2]:
import sentencepiece as spm

In [4]:
# write a toy.txt file with some random text
with open("toy.txt", "w", encoding="utf-8") as f:
  f.write("SentencePiece is an unsupervised text tokenizer and detokenizer mainly for Neural Network-based text generation systems where the vocabulary size is predetermined prior to the neural model training. SentencePiece implements subword units (e.g., byte-pair-encoding (BPE) [Sennrich et al.]) and unigram language model [Kudo.]) with the extension of direct training from raw sentences. SentencePiece allows us to make a purely end-to-end system that does not depend on language-specific pre/postprocessing.")

byte_fallback指标可以把未知的进行映射，sentencepiece可以将其回退到utf-8来处理稀有码点，否则都映射到unk；

add_dummy_prefix会在文本开头添加虚拟空白，以便以完全相同的方式来应对单词，比如"world"和"hello world"中的world，这里就都统一为前面有空格

In [5]:
# train a sentencepiece model on it
# the settings here are (best effort) those used for training Llama 2
import os

options = dict(
  # input spec
  input="toy.txt",
  input_format="text",
  # output spec
  model_prefix="tok400", # output filename prefix
  # algorithm spec
  # BPE alg
  model_type="bpe",
  vocab_size=400,
  # normalization
  normalization_rule_name="identity", # ew, turn off normalization
  remove_extra_whitespaces=False,
  input_sentence_size=200000000, # max number of training sentences
  max_sentence_length=4192, # max number of bytes per sentence
  seed_sentencepiece_size=1000000,
  shuffle_input_sentence=True,
  # rare word treatment
  character_coverage=0.99995,
  byte_fallback=True,
  # merge rules
  split_digits=True,
  split_by_unicode_script=True,
  split_by_whitespace=True,
  split_by_number=True,
  max_sentencepiece_length=16,
  add_dummy_prefix=True,
  allow_whitespace_only_pieces=True,
  # special tokens
  unk_id=0, # the UNK token MUST exist
  bos_id=1, # the others are optional, set to -1 to turn off
  eos_id=2,
  pad_id=-1,
  # systems
  num_threads=os.cpu_count(), # use ~all system resources
)

spm.SentencePieceTrainer.train(**options)

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: toy.txt
  input_format: text
  model_prefix: tok400
  model_type: BPE
  vocab_size: 400
  self_test_sample_size: 0
  character_coverage: 0.99995
  input_sentence_size: 200000000
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 192
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 1
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 1
  required_chars: 
  byte_fallback: 1
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  dif

In [6]:
sp = spm.SentencePieceProcessor()
sp.load('tok400.model')
vocab = [[sp.id_to_piece(idx), idx] for idx in range(sp.get_piece_size())]
vocab

[['<unk>', 0],
 ['<s>', 1],
 ['</s>', 2],
 ['<0x00>', 3],
 ['<0x01>', 4],
 ['<0x02>', 5],
 ['<0x03>', 6],
 ['<0x04>', 7],
 ['<0x05>', 8],
 ['<0x06>', 9],
 ['<0x07>', 10],
 ['<0x08>', 11],
 ['<0x09>', 12],
 ['<0x0A>', 13],
 ['<0x0B>', 14],
 ['<0x0C>', 15],
 ['<0x0D>', 16],
 ['<0x0E>', 17],
 ['<0x0F>', 18],
 ['<0x10>', 19],
 ['<0x11>', 20],
 ['<0x12>', 21],
 ['<0x13>', 22],
 ['<0x14>', 23],
 ['<0x15>', 24],
 ['<0x16>', 25],
 ['<0x17>', 26],
 ['<0x18>', 27],
 ['<0x19>', 28],
 ['<0x1A>', 29],
 ['<0x1B>', 30],
 ['<0x1C>', 31],
 ['<0x1D>', 32],
 ['<0x1E>', 33],
 ['<0x1F>', 34],
 ['<0x20>', 35],
 ['<0x21>', 36],
 ['<0x22>', 37],
 ['<0x23>', 38],
 ['<0x24>', 39],
 ['<0x25>', 40],
 ['<0x26>', 41],
 ['<0x27>', 42],
 ['<0x28>', 43],
 ['<0x29>', 44],
 ['<0x2A>', 45],
 ['<0x2B>', 46],
 ['<0x2C>', 47],
 ['<0x2D>', 48],
 ['<0x2E>', 49],
 ['<0x2F>', 50],
 ['<0x30>', 51],
 ['<0x31>', 52],
 ['<0x32>', 53],
 ['<0x33>', 54],
 ['<0x34>', 55],
 ['<0x35>', 56],
 ['<0x36>', 57],
 ['<0x37>', 58],
 ['<0x38>', 5

In [7]:
ids = sp.encode("hello 안녕하세요")
print(ids)

[362, 378, 361, 372, 358, 362, 239, 152, 139, 238, 136, 152, 240, 152, 155, 239, 135, 187, 239, 157, 151]


In [8]:
print([sp.id_to_piece(idx) for idx in ids])

['▁', 'h', 'e', 'l', 'lo', '▁', '<0xEC>', '<0x95>', '<0x88>', '<0xEB>', '<0x85>', '<0x95>', '<0xED>', '<0x95>', '<0x98>', '<0xEC>', '<0x84>', '<0xB8>', '<0xEC>', '<0x9A>', '<0x94>']


### 设计词汇表vocab_size大小需要考虑

1、以gpt.py文件为例，vocab_size越大，意味着行数越多，每一行都需要在最后进行softmax计算获得概率值，计算量就越大

2、vocab_size变大，token数量就会减少，意味着每个token涵盖的信息量会更多

### tokenization

1、为什么LLMs在非英语语言上表现较差？非英文训练数据较少；tokenizer也没有在非英文数据上得到充分训练

2、LLMs为什么不擅长简单算术运算？与数字的tokenization有关

3、SolidGoldMargikarp：在tokenizer训练过程中可能某个用户名为这个的，发了很多次内容使得tokenizer学习到了很多相关信息，就记住了这个单词。但在LLMs中，它从未在前向训练、反向传播当中学习到这个词的有关信息，也就是超过了样本范围和分布范围，使得用户向它输入时，无法辨别。

4、我们倾向于使用yaml而不是json，因为长度更短编码更高效